#Telecom Domain ReadOps Assignment
This notebook contains assignments to practice Spark read options and Databricks volumes. <br>
Sections: Sample data creation, Catalog & Volume creation, Copying data into Volumes, Path glob/recursive reads, toDF() column renaming variants, inferSchema/header/separator experiments, and exercises.<br>

![](https://fplogoimages.withfloats.com/actual/68009c3a43430aff8a30419d.png)
![](https://theciotimes.com/wp-content/uploads/2021/03/TELECOM1.jpg)

##First Import all required libraries & Create spark session object

In [0]:
from pyspark.sql.session import SparkSession
print(spark)#default databricks session instantiated
spark1 = SparkSession.builder.getOrCreate()
print(spark1)#user instantiated spark object, both refers to same object

##1. Write SQL statements to create:
1. A catalog named telecom_catalog_assign
2. A schema landing_zone
3. A volume landing_vol
4. Using dbutils.fs.mkdirs, create folders:<br>
/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/
/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/
/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/
5. Explain the difference between (Just google and understand why we are going for volume concept for prod ready systems):<br>
a. Volume vs DBFS/FileStore<br>
b. Why production teams prefer Volumes for regulated data<br>

In [0]:
%sql
create catalog if not exists telecom_catalog_assign;
create database if not exists telecom_catalog_assign.landing_zone;
create volume if not exists telecom_catalog_assign.landing_zone.landing_vol;

In [0]:
dbutils.fs.mkdirs("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/")
dbutils.fs.mkdirs("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/")
dbutils.fs.mkdirs("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/")

##Data files to use in this usecase:
customer_csv = '''
101,Arun,31,Chennai,PREPAID
102,Meera,45,Bangalore,POSTPAID
103,Irfan,29,Hyderabad,PREPAID
104,Raj,52,Mumbai,POSTPAID
105,,27,Delhi,PREPAID
106,Sneha,abc,Pune,PREPAID
'''

usage_tsv = '''customer_id\tvoice_mins\tdata_mb\tsms_count
101\t320\t1500\t20
102\t120\t4000\t5
103\t540\t600\t52
104\t45\t200\t2
105\t0\t0\t0
'''

tower_logs_region1 = '''event_id|customer_id|tower_id|signal_strength|timestamp
5001|101|TWR01|-80|2025-01-10 10:21:54
5004|104|TWR05|-75|2025-01-10 11:01:12
'''

# **Data files to use in this usecase:**
customer_csv = ''' 101,Arun,31,Chennai,PREPAID <br>
102,Meera,45,Bangalore,POSTPAID <br>
103,Irfan,29,Hyderabad,PREPAID <br>
104,Raj,52,Mumbai,POSTPAID <br>
105,,27,Delhi,PREPAID <br>
106,Sneha,abc,Pune,PREPAID '''

usage_tsv = '''customer_id\tvoice_mins\tdata_mb\tsms_count <br>
101\t320\t1500\t20 <br>
102\t120\t4000\t5 <br>
103\t540\t600\t52 <br>
104\t45\t200\t2 <br>
105\t0\t0\t0 '''

tower_logs_region1 = '''event_id|customer_id|tower_id|signal_strength|timestamp <br>
5001|101|TWR01|-80|2025-01-10 10:21:54 <br>
5004|104|TWR05|-75|2025-01-10 11:01:12 '''

##2. Filesystem operations
1. Write code to copy the above datasets into your created Volume folders:
Customer → /Volumes/.../customer/
Usage → /Volumes/.../usage/
Tower (region-based) → /Volumes/.../tower/region1/ and /Volumes/.../tower/region2/

2. Write a command to validate whether files were successfully copied

In [0]:
customer_csv = """ 101,Arun,31,Chennai,PREPAID 
102,Meera,45,Bangalore,POSTPAID 
103,Irfan,29,Hyderabad,PREPAID 
104,Raj,52,Mumbai,POSTPAID 
105,,27,Delhi,PREPAID 
106,Sneha,abc,Pune,PREPAID """

usage_tsv = """customer_id\tvoice_mins\tdata_mb\tsms_count 
101\t320\t1500\t20 
102\t120\t4000\t5 
103\t540\t600\t52 
104\t45\t200\t2 
105\t0\t0\t0 """

tower_logs_region1 = """event_id|customer_id|tower_id|signal_strength|timestamp 
5001|101|TWR01|-80|2025-01-10 10:21:54 
5004|104|TWR05|-75|2025-01-10 11:01:12 """

dbutils.fs.put("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer.csv", customer_csv, True)
dbutils.fs.put("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/usage.csv", usage_tsv, True)
dbutils.fs.put("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/tower_logs_region1", tower_logs_region1, True)

In [0]:
#move file from tower to region1 dir
dbutils.fs.cp("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region1/","/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region2/",True)

##3. Directory Read Use Cases
1. Read all tower logs using:
Path glob filter (example: *.csv)
Multiple paths input
Recursive lookup

2. Demonstrate these 3 reads separately:
Using pathGlobFilter
Using list of paths in spark.read.csv([path1, path2])
Using .option("recursiveFileLookup","true")

3. Compare the outputs and understand when each should be used.

In [0]:
#recursiveFileLookup=True, reads files from the subfolders too
#pathGlobFilter="tower_logs_*", reads files with the pattern starts with tower_logs_, if file name unknown, we can use *.csv
df_multiple_path_files=spark.read.csv(path=["/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region1","/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region2"],header=True,inferSchema=True,sep="|",recursiveFileLookup=True,pathGlobFilter="tower_logs_*")
print(df_multiple_path_files.count())


##4. Schema Inference, Header, and Separator
1. Try the Customer, Usage files with the option and options using read.csv and format function:<br>
header=false, inferSchema=false<br>
or<br>
header=true, inferSchema=true<br>
2. Write a note on What changed when we use header or inferSchema  with true/false?<br>
3. How schema inference handled “abc” in age?<br>

In [0]:
'''Try the Customer, Usage files with the option and options using read.csv and format function:
header=false, inferSchema=false
or
header=true, inferSchema=true'''
df1 = spark1.read.options(header="false", inferSchema="false").csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer.csv")
df1.printSchema
display(df1)

- while using header=True, its taking 1st row as a column name, in case of False, its taking c0,c1,c2, so on by default
- if we use toDF with header=False, it will take user specified column name instead of default
- if we use toDF with header=True, it will take user specified column name instead of default but 1st row is getting vanished - not recommended (leads to data loss)
- inferSchema=True - based on data, it will assign the column datatype
- inferSchema=False - by default string is a datatype

### output
> Age datatype considered as String as age of Sneha mentioned as "abc", if not str else number, datatype will be an integer. <br>
> Null is present in the custname.


##5. Column Renaming Usecases
1. Apply column names using string using toDF function for customer data
2. Apply column names and datatype using the schema function for usage data
3. Apply column names and datatype using the StructType with IntegerType, StringType, TimestampType and other classes for towers data 

In [0]:
#1. Apply column names using string using toDF function for customer data
df1 = spark.read.options(inferSchema="true").format("csv").load("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/customer.csv").toDF("customer_id","name","age","city","plan")
display(df1)

#2. Apply column names and datatype using the schema function for usage data
str_struct="customer_id integer, voice_mins integer, data_mb integer, sms_count string"
use_schema=spark.read.schema(str_struct).options(header=True,sep="\t").csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/usage/usage.csv")
display(use_schema)

#3. Apply column names and datatype using the StructType with IntegerType, StringType, TimestampType and other classes for towers data
from pyspark.sql.types import StructType,StructField,IntegerType,StringType
cust_schema=StructType(
    [
        StructField("event_id",IntegerType(),True),
        StructField("customer_id",IntegerType(),True),
        StructField("tower_id",StringType(),True),
        StructField("signal_strength",StringType(),True),
        StructField("timestamp",StringType(),True)
    ]
)
df_cust_schema=spark.read.schema(cust_schema).options(header=True,sep="|").format("csv").load("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/tower/region1/tower_logs_region1")
display(df_cust_schema)

## 6. More to come (stay motivated)....